# NLP Extractive Text Summarization
### Sistem Bilingual — Inggris dan Indonesia
---
**Deskripsi:**
Notebook ini mengimplementasi sistem Extractive Text Summarization untuk dokumen berbahasa Indonesia dan Inggris.
Metode representasi menggunakan TF-IDF and Word2Vec, evaluasi dengan ROUGE

**Library:** NLTK, spaCy, PySastrawi, Gensim, Scikit-learn, Matplotlib, WordCloud

## 1. Setup, Instalasi, dan Import Library

In [1]:
# Setup dan Instalasi Library
!pip install PySastrawi
!pip install rouge-score
!pip install wordcloud
!pip install langdetect
!pip install nltk
!pip install gensim

In [2]:
# Import semua Library
# Standard libraries
import os
import re
import json
import string
# Data handling
import pandas as pd
import numpy as np
# NLP — General
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
# NLP — Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
# NLP — Inggris
import spacy
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec
# Visualization
import matplotlib.pyplot as plt
from wordcloud import WordCloud
# Evaluation
from rouge_score import rouge_scorer
# Language detection
from langdetect import detect
# NLTK downloads
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
# Download model bahasa inggris spacy
!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
print("spaCy English model loaded successfully.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Masukkan dan Load Dataset dari Kaggle

In [ ]:
# Loading Dataset
from google.colab import files
files.upload()

In [ ]:
# Update kaggle and move kaggle.json to directory
!pip install --upgrade kaggle
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("Kaggle credentials configured.")

In [ ]:
# Download IndoSum (Artikel Bahasa Indonesia)
!kaggle datasets download -d linkgish/indosum --unzip -p /content/indosum

# Download BBC News Summary (Artikel Bahasa Inggris)
!kaggle datasets download -d pariza/bbc-news-summary --unzip -p /content/bbc

In [ ]:
# Cek Struktur File
print("=== IndoSum Files ===")
for root, dirs, files_list in os.walk("/content/indosum"):
    for f in files_list:
        print(os.path.join(root, f))

print("\n=== BBC Files (first 20) ===")
count = 0
for root, dirs, files_list in os.walk("/content/bbc"):
    for f in files_list:
        if count < 20:
            print(os.path.join(root, f))
            count += 1

### 2a. Loading dataset

In [ ]:
# Load IndoSum
import glob

# Find all jsonl files
jsonl_files = glob.glob("/content/indosum/**/*.jsonl", recursive=True)
print("Found JSONL files:", jsonl_files)

# Load all into one dataframe
indo_records = []
for filepath in jsonl_files:
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            indo_records.append(record)

df_indo = pd.DataFrame(indo_records)
print(f"\nTotal Indonesian documents loaded: {len(df_indo)}")
print(f"Columns: {df_indo.columns.tolist()}")
df_indo.head(3)

In [ ]:
#Load BBC News
bbc_records = []

articles_path = "/content/bbc/BBC News Summary/News Articles"
summaries_path = "/content/bbc/BBC News Summary/Summaries"

categories = os.listdir(articles_path)
print("Categories found:", categories)

for category in categories:
    article_folder = os.path.join(articles_path, category)
    summary_folder = os.path.join(summaries_path, category)

    if not os.path.isdir(article_folder):
        continue

    for filename in os.listdir(article_folder):
        article_file = os.path.join(article_folder, filename)
        summary_file = os.path.join(summary_folder, filename)

        try:
            with open(article_file, "r", encoding="utf-8", errors="ignore") as af:
                article_text = af.read().strip()
            with open(summary_file, "r", encoding="utf-8", errors="ignore") as sf:
                summary_text = sf.read().strip()

            bbc_records.append({
                "category": category,
                "article": article_text,
                "summary": summary_text,
                "filename": filename
            })
        except FileNotFoundError:
            continue  # Skip if no matching summary file

df_bbc = pd.DataFrame(bbc_records)
print(f"\nTotal English documents loaded: {len(df_bbc)}")
print(f"Columns: {df_bbc.columns.tolist()}")
df_bbc.head(3)

## 3.  Analisis Dataset, dan Pengambilan sampel untuk preprocessing

In [ ]:
# Analisis dan eksplorasi dataset
print("=" * 50)
print("INDOSUM — INDONESIAN DATASET")
print("=" * 50)
print(f"Total documents    : {len(df_indo)}")
print(f"Columns            : {df_indo.columns.tolist()}")
print(f"Missing values     :\n{df_indo.isnull().sum()}")

print("\n" + "=" * 50)
print("BBC NEWS — ENGLISH DATASET")
print("=" * 50)
print(f"Total documents    : {len(df_bbc)}")
print(f"Category breakdown :\n{df_bbc['category'].value_counts()}")
print(f"Missing values     :\n{df_bbc.isnull().sum()}")

In [ ]:
# Standardisasi dataset

def flatten_paragraphs(paragraphs):
    """
    Handles IndoSum's 3-level nested structure:
    paragraphs → list of paragraphs → list of sentences → list of words
    """
    if not isinstance(paragraphs, list):
        return str(paragraphs)

    sentences = []
    for paragraph in paragraphs:
        for sentence in paragraph:
            if isinstance(sentence, list):
                # sentence is a list of words
                sentences.append(" ".join(sentence))
            else:
                # sentence is already a string
                sentences.append(sentence)
    return " ".join(sentences)

def flatten_summary(summary):
    """
    Handles IndoSum's summary field:
    summary → list of sentences → list of words (or list of strings)
    """
    if not isinstance(summary, list):
        return str(summary)

    sentences = []
    for sentence in summary:
        if isinstance(sentence, list):
            sentences.append(" ".join(sentence))
        else:
            sentences.append(sentence)
    return " ".join(sentences)


# Apply to IndoSum
df_indo_clean = pd.DataFrame({
    "article" : df_indo["paragraphs"].apply(flatten_paragraphs),
    "summary" : df_indo["summary"].apply(flatten_summary),
    "language": "indonesian"
})

# BBC
df_bbc_clean = pd.DataFrame({
    "article" : df_bbc["article"],
    "summary" : df_bbc["summary"],
    "language": "english"
})

# Combine
df_all = pd.concat([df_indo_clean, df_bbc_clean], ignore_index=True)

# Print a sample from each
print(f"Combined dataset: {len(df_all)} documents")
print(f"\nLanguage breakdown:\n{df_all['language'].value_counts()}")

print("\n=== Artikel Indosum (200 CHR) ===")
print(df_all[df_all["language"] == "indonesian"]["article"].iloc[0][:200])

print("\n=== Artikel BBC (200 CHR) ===")
print(df_all[df_all["language"] == "english"]["article"].iloc[0][:200])

In [ ]:
# Pecah sampel bahasa indonesia menjadi 3000 agar seimbang dengan sampel inggris
df_indo_clean = df_indo_clean.sample(n=3000, random_state=42).reset_index(drop=True)

df_all = pd.concat([df_indo_clean, df_bbc_clean], ignore_index=True).reset_index(drop=True)

print(f"Rebalanced dataset: {len(df_all)} documents")
print(df_all["language"].value_counts())

## 4. Text Preprocessing
cleaning → tokenization → stopword removal → stemming/lemmatization.





In [ ]:
#Cleaning
def clean_text(text):
    """Cleaning pada kedua jenis artikel sebelum masuk ke bagian spesifik per bahasa"""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)        # Remove URLs
    text = re.sub(r"\[.*?\]", "", text)                # Remove bracketed content
    text = re.sub(r"[^a-zA-Z\s]", " ", text)           # Remove numbers & punctuation
    text = re.sub(r"\s+", " ", text).strip()            # Collapse extra whitespace
    return text

df_all["article_clean"] = df_all["article"].apply(clean_text)
df_all["summary_clean"] = df_all["summary"].apply(clean_text)

print("Cleaning Selesai")
print("\nSample cleaned Indonesian article:")
print(df_all[df_all["language"]=="indonesian"]["article_clean"].iloc[0][:200])
print("\nSample cleaned English article:")
print(df_all[df_all["language"]=="english"]["article_clean"].iloc[0][:200])

In [ ]:
# Tokenisasi (Kedua Bahasa)
def tokenize(text):
    return word_tokenize(text)

df_all["tokens"] = df_all["article_clean"].apply(tokenize)

print("Tokenization done.")
print("\nSample Indonesian tokens (15 token):")
print(df_all[df_all["language"]=="indonesian"]["tokens"].iloc[0][:15])
print("\nSample English tokens (15 token):")
print(df_all[df_all["language"]=="english"]["tokens"].iloc[0][:15])

In [ ]:
# Stopword Removal (Beda untuk kedua bahasa)
# English stopwords from NLTK ---
english_stops = set(stopwords.words("english"))

# Indonesian stopwords from PySastrawi ---
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
indo_stop_factory = StopWordRemoverFactory()
indo_stops = set(indo_stop_factory.get_stop_words())

def remove_stopwords(tokens, language):
    if language == "english":
        return [t for t in tokens if t not in english_stops and len(t) > 1]
    else:
        return [t for t in tokens if t not in indo_stops and len(t) > 1]

df_all["tokens_no_stop"] = df_all.apply(
    lambda row: remove_stopwords(row["tokens"], row["language"]), axis=1
)

print("Stopword removal done.")
print(f"\nEnglish stopwords sample: {list(english_stops)[:10]}")
print(f"Indonesian stopwords sample: {list(indo_stops)[:10]}")

print("\nSample Indonesian (15 pertama):")
print(df_all[df_all["language"]=="indonesian"]["tokens_no_stop"].iloc[0][:15])
print("\nSample English (15 pertama):")
print(df_all[df_all["language"]=="english"]["tokens_no_stop"].iloc[0][:15])

In [ ]:
# Stemming dan Lemmatisasi
# Indonesian Stemmer (PySastrawi) ---
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
stem_factory = StemmerFactory()
indo_stemmer = stem_factory.create_stemmer()

# English Lemmatizer (spaCy) ---
nlp = spacy.load("en_core_web_sm")

def stem_indonesian(tokens):
    return [indo_stemmer.stem(token) for token in tokens]

def lemmatize_english(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

# Apply (Bisa memakan waktu beberapa menit untuk 5000 dokumen)
print("Applying stemming/lemmatization... (this may take a few minutes)")

df_indo_mask = df_all["language"] == "indonesian"
df_eng_mask  = df_all["language"] == "english"

df_all.loc[df_indo_mask, "tokens_final"] = \
    df_all.loc[df_indo_mask, "tokens_no_stop"].apply(stem_indonesian)

df_all.loc[df_eng_mask, "tokens_final"] = \
    df_all.loc[df_eng_mask, "tokens_no_stop"].apply(lemmatize_english)

print("Done.")
print("\nSample Indonesian tokens (final):")
print(df_all[df_all["language"]=="indonesian"]["tokens_final"].iloc[0][:15])
print("\nSample English tokens (final):")
print(df_all[df_all["language"]=="english"]["tokens_final"].iloc[0][:15])

In [ ]:
# Rekonstruksi
# Rejoin tokens into processed strings
df_all["text_processed"] = df_all["tokens_final"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

# Drop any empty results
before = len(df_all)
df_all = df_all[df_all["text_processed"].str.strip() != ""].reset_index(drop=True)
after  = len(df_all)
print(f"Dropped {before - after} empty documents after preprocessing.")

# Final overview
print(f"\nFinal dataset size: {len(df_all)}")
print(f"\nColumn overview:")
print(df_all[["language","article","article_clean","text_processed"]].head(3).to_string())

## 5. Representasi Vektor
Metode yang digunakan:
- **TF-IDF**
- **Word2Vec**

In [ ]:
# Pemisahan Artikel untuk vektorisasi
df_indo_proc = df_all[df_all["language"] == "indonesian"].reset_index(drop=True)
df_eng_proc  = df_all[df_all["language"] == "english"].reset_index(drop=True)

print(f"Indonesian documents : {len(df_indo_proc)}")
print(f"English documents    : {len(df_eng_proc)}")

### 5a. Metode 1: TF-IDF (Term Frequency-Inverse Document Frequency)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Indonesian TF-IDF ---
tfidf_indo = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix_indo = tfidf_indo.fit_transform(df_indo_proc["text_processed"])

# English TF-IDF ---
tfidf_eng = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix_eng = tfidf_eng.fit_transform(df_eng_proc["text_processed"])

print("=== TF-IDF Matrix Shapes ===")
print(f"Indonesian : {tfidf_matrix_indo.shape}  → (documents × vocabulary)")
print(f"English    : {tfidf_matrix_eng.shape}")

# Preview top 15
def top_tfidf_terms(vectorizer, matrix, n=15):
    feature_names = vectorizer.get_feature_names_out()
    mean_scores   = matrix.mean(axis=0).A1
    top_indices   = mean_scores.argsort()[::-1][:n]
    return [(feature_names[i], round(mean_scores[i], 4)) for i in top_indices]

print("\n--- Top 15 Indonesian TF-IDF Terms ---")
for term, score in top_tfidf_terms(tfidf_indo, tfidf_matrix_indo):
    print(f"  {term:<30} {score}")

print("\n--- Top 15 English TF-IDF Terms ---")
for term, score in top_tfidf_terms(tfidf_eng, tfidf_matrix_eng):
    print(f"  {term:<30} {score}")

In [ ]:
# Scoring untuk TF-IDF dan Contoh Summarization
def tfidf_summarize(article_raw, vectorizer, top_n=3):
    """
    Scores each sentence in an article using TF-IDF word weights,
    then returns the top_n sentences as the extractive summary.
    """
    # Split into sentences
    sentences = sent_tokenize(article_raw)
    if len(sentences) <= top_n:
        return " ".join(sentences)

    # Get vocabulary and IDF weights from the fitted vectorizer
    vocab      = vectorizer.vocabulary_
    idf_vals   = vectorizer.idf_

    # Score each sentence
    scores = []
    for sent in sentences:
        words  = word_tokenize(clean_text(sent))
        score  = sum(idf_vals[vocab[w]] for w in words if w in vocab)
        scores.append(score)

    # Pick top_n sentence indices (preserving original order)
    top_indices = sorted(
        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    )
    return " ".join([sentences[i] for i in top_indices])


# Quick test on one Indonesian and one English article
sample_indo_summary = tfidf_summarize(
    df_indo_proc["article"].iloc[0], tfidf_indo, top_n=3
)
sample_eng_summary = tfidf_summarize(
    df_eng_proc["article"].iloc[0], tfidf_eng, top_n=3
)

print("=== TF-IDF Extractive Summary — Indonesian ===")
print(sample_indo_summary)
print("\n=== TF-IDF Extractive Summary — English ===")
print(sample_eng_summary)

In [ ]:
# Pembuatan TF-IDF untuk semua artikel
print("Generating TF-IDF summaries for all documents...")

df_indo_proc["summary_tfidf"] = df_indo_proc["article"].apply(
    lambda x: tfidf_summarize(x, tfidf_indo, top_n=3)
)
df_eng_proc["summary_tfidf"] = df_eng_proc["article"].apply(
    lambda x: tfidf_summarize(x, tfidf_eng, top_n=3)
)

print(f"Indonesian summaries generated: {len(df_indo_proc)}")
print(f"English summaries generated   : {len(df_eng_proc)}")

### 5b. Metode 2: Word2Vec

In [ ]:
from gensim.models import Word2Vec

# Prepare token lists (Word2Vec expects a list of token lists)
indo_token_corpus = df_indo_proc["tokens_final"].tolist()
eng_token_corpus  = df_eng_proc["tokens_final"].tolist()

# --- Train Indonesian Word2Vec ---
w2v_indo = Word2Vec(
    sentences  = indo_token_corpus,
    vector_size= 100,       # Embedding dimensions
    window     = 5,         # Context window size
    min_count  = 2,         # Ignore words appearing fewer than 2 times
    workers    = 4,
    sg         = 1,         # 1 = Skip-gram, 0 = CBOW
    epochs     = 10
)

# --- Train English Word2Vec ---
w2v_eng = Word2Vec(
    sentences  = eng_token_corpus,
    vector_size= 100,
    window     = 5,
    min_count  = 2,
    workers    = 4,
    sg         = 1,
    epochs     = 10
)

print("Word2Vec training complete.")
print(f"Indonesian vocabulary size : {len(w2v_indo.wv)}")
print(f"English vocabulary size    : {len(w2v_eng.wv)}")

# Quick sanity check — most similar words
print("\n--- Indonesian: words similar to 'jakarta' ---")
print(w2v_indo.wv.most_similar("jakarta", topn=5))

print("\n--- English: words similar to 'government' ---")
print(w2v_eng.wv.most_similar("government", topn=5))

In [ ]:
# Scoring untuk Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

def get_sentence_vector(tokens, wv_model):
    """Average the word vectors for a list of tokens."""
    vectors = [wv_model[w] for w in tokens if w in wv_model]
    if not vectors:
        return np.zeros(wv_model.vector_size)
    return np.mean(vectors, axis=0)


def w2v_summarize(article_raw, wv_model, top_n=3):
    """
    Scores sentences by cosine similarity to the document centroid vector,
    then returns the top_n most central sentences as the extractive summary.
    """
    sentences = sent_tokenize(article_raw)
    if len(sentences) <= top_n:
        return " ".join(sentences)

    # Tokenize and vectorize each sentence
    tokenized = [word_tokenize(clean_text(s)) for s in sentences]
    sent_vecs = np.array([get_sentence_vector(t, wv_model.wv) for t in tokenized])

    # Document centroid = mean of all sentence vectors
    centroid  = sent_vecs.mean(axis=0, keepdims=True)

    # Cosine similarity of each sentence to centroid
    sims      = cosine_similarity(sent_vecs, centroid).flatten()

    # Pick top_n, preserving document order
    top_indices = sorted(
        sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)[:top_n]
    )
    return " ".join([sentences[i] for i in top_indices])


# Quick test
sample_indo_w2v = w2v_summarize(df_indo_proc["article"].iloc[0], w2v_indo, top_n=3)
sample_eng_w2v  = w2v_summarize(df_eng_proc["article"].iloc[0],  w2v_eng,  top_n=3)

print("=== Word2Vec Summary — Indonesian ===")
print(sample_indo_w2v)
print("\n=== Word2Vec Summary — English ===")
print(sample_eng_w2v)

In [ ]:
# Pembuatan Word2Vec untuk semua artikel
print("Generating Word2Vec summaries... (may take a few minutes)")

df_indo_proc["summary_w2v"] = df_indo_proc["article"].apply(
    lambda x: w2v_summarize(x, w2v_indo, top_n=3)
)
df_eng_proc["summary_w2v"] = df_eng_proc["article"].apply(
    lambda x: w2v_summarize(x, w2v_eng, top_n=3)
)

print("Done.")
print("\nSample columns now available:")
print(df_indo_proc[["article", "summary", "summary_tfidf", "summary_w2v"]].iloc[0])

## 6. Visualisasi Menggunakan 4 Bentuk
1. Word Clouds
2. TF-IDF Top Terms Bar Plot
3. PCA on Word2Vec Embeddings
4. ROUGE Score: Evaluasi antara 2 metode vektor

### 6a. WordCloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def generate_wordcloud(text_series, title, colormap="viridis"):
    """
    Joins all preprocessed documents into one string
    and renders a word cloud from term frequencies.
    """
    full_corpus = " ".join(text_series.dropna().tolist())

    wc = WordCloud(
        width           = 1000,
        height          = 500,
        background_color= "white",
        colormap        = colormap,
        max_words       = 150,
        collocations    = False   # Avoids duplicate bigrams
    ).generate(full_corpus)

    plt.figure(figsize=(14, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

# Indonesian word cloud
generate_wordcloud(
    df_indo_proc["text_processed"],
    "Word Cloud — Indonesian Corpus (Post-Preprocessing)",
    colormap="YlOrRd"
)

# English word cloud
generate_wordcloud(
    df_eng_proc["text_processed"],
    "Word Cloud — English Corpus (Post-Preprocessing)",
    colormap="Blues"
)